In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config
from src.console import print_header, print_kv, print_status
from src.error_analysis import load_run_predictions, save_representative_utterance_artifacts
from src.model_analysis import (aggregate_permutation_importance_by_group, compute_permutation_importance,
                                compute_shap_values, evaluate_branch_ablation, load_run_embeddings,
                                load_run_gate_weights, plot_branch_ablation, plot_complementarity_heatmap,
                                plot_embedding_map, plot_gate_distribution, plot_permutation_importance,
                                plot_shap_group_importance, plot_shap_summary, summarize_gate_values)
from src.praat import FEATURE_GROUPS, SEGMENTAL_FEATURE_COLUMNS, SUPRASEGMENTAL_FEATURE_COLUMNS, load_praat_table
from src.results import (build_dataset_table, build_feature_architecture_table, build_final_metrics_table,
                         build_model_dimensions_table, build_per_class_metrics_table, export_paper_tables,
                         select_analysis_run)
from src.splits import iter_severity_loso_folds
from src.training.checkpoint import load_checkpoint
from src.training.data import build_loaders, load_manifest
from src.training.metrics import compute_confusion_matrix
from src.training.models import SEVERITY_MODEL_NAME, build_model
from src.training.reporting import save_confusion_matrix
from src.training.utils import resolve_device

config.ensure_directories()
TASK = "severity"

df_m6 = load_manifest()
RUN_NAME = select_analysis_run(task=TASK, metric="f1", preferred="severity_gated_fusion_three_branch")
device = resolve_device(None)

print_header("Three-Branch Severity Architecture -- Final Results")
print_kv("Analysis run", RUN_NAME or "none eligible yet -- run notebooks/03_training.ipynb first")

In [ ]:
dataset_table = build_dataset_table(df_m6)
final_metrics_table = None
if RUN_NAME is not None:
    try:
        final_metrics_table = build_final_metrics_table(RUN_NAME)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
print_kv("Dataset table rows", len(dataset_table))
final_metrics_table

In [ ]:
per_class_table, preds = None, None
if RUN_NAME is not None:
    try:
        preds = load_run_predictions(RUN_NAME)
        per_class_table = build_per_class_metrics_table(RUN_NAME, task=TASK)
        cm = compute_confusion_matrix(preds["y_true"].to_numpy(), preds["y_pred"].to_numpy(), TASK)
        save_confusion_matrix(config.METRIC_FIGURE_DIR / f"{RUN_NAME}_final_confusion_matrix.png",
                              cm, TASK, title=f"{RUN_NAME} -- final confusion matrix")
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
per_class_table

In [ ]:
ablation_df = None
if RUN_NAME is not None:
    ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME
    fold_dirs = sorted(p.parent.name for p in ckpt_dir.glob("*/best.pt")) if ckpt_dir.exists() else []
    if fold_dirs:
        model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES[TASK], num_speakers=1).to(device)
        load_checkpoint(ckpt_dir / fold_dirs[0] / "best.pt", model, map_location=str(device))
        model.eval()
        _, train_df, test_df = next(iter(iter_severity_loso_folds(df_m6)))
        _, _, test_loader = build_loaders(train_df, train_df.head(0), test_df, batch_size=8,
                                          num_workers=0, pin_memory=False, model_name=SEVERITY_MODEL_NAME)
        ablation_df = evaluate_branch_ablation(model, test_loader, device, task=TASK)
        plot_branch_ablation(ablation_df, metric="macro_f1", show=True)
ablation_df

In [ ]:
gate_summary_df = None
if RUN_NAME is not None:
    try:
        gate_df = load_run_gate_weights(RUN_NAME)
        gate_summary_df = summarize_gate_values(gate_df)
        plot_gate_distribution(gate_df, show=True)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
gate_summary_df

In [ ]:
shap_group_df, permutation_group_df = None, None
try:
    praat_table = load_praat_table().reset_index()
    feature_columns = list(SEGMENTAL_FEATURE_COLUMNS) + list(SUPRASEGMENTAL_FEATURE_COLUMNS)
    shap_explanations, X_sample, shap_feature_columns, class_names, surrogate = compute_shap_values(
        praat_table, task=TASK, feature_columns=feature_columns, run_name=RUN_NAME)
    plot_shap_summary(shap_explanations[-1], shap_feature_columns, show=True,
                      title=f"SHAP -- {class_names[-1]} class (surrogate model)")

    global_mean_abs = np.mean([np.abs(e.values).mean(axis=0) for e in shap_explanations], axis=0)
    global_table = pd.DataFrame({"feature": shap_feature_columns, "mean_abs_shap": global_mean_abs})
    global_table["group"] = global_table["feature"].map(FEATURE_GROUPS)
    shap_group_df = (global_table.groupby("group")["mean_abs_shap"].sum()
                     .sort_values(ascending=False).reset_index())
    plot_shap_group_importance(shap_group_df, show=True)

    label_map = {k: v for k, v in config.SEVERITY_LABEL_MAP.items() if v >= 0}
    labeled = praat_table[praat_table["Severity"].isin(label_map)].dropna(subset=shap_feature_columns)
    permutation_df = compute_permutation_importance(
        surrogate, labeled[shap_feature_columns].to_numpy(),
        labeled["Severity"].map(label_map).to_numpy(), shap_feature_columns)
    permutation_group_df = aggregate_permutation_importance_by_group(permutation_df)
    plot_permutation_importance(permutation_df, show=True)
except FileNotFoundError as e:
    print_status(str(e), ok=False)

In [ ]:
if RUN_NAME is not None and preds is not None:
    for embedding_type in ("learned", "segmental", "supra", "fused"):
        try:
            plot_embedding_map(RUN_NAME, preds, task=TASK, method="pca", embedding_type=embedding_type,
                               color_by="severity", show=True)
        except (FileNotFoundError, ValueError) as e:
            print_status(f"{embedding_type} (severity, PCA): {e}", ok=False)
    try:
        import umap  # noqa: F401
        plot_embedding_map(RUN_NAME, preds, task=TASK, method="umap", embedding_type="fused",
                           color_by="severity", show=True)
    except ImportError:
        print_status("umap-learn not installed -- skipping UMAP.", ok=False)

In [ ]:
representative_model = None
if RUN_NAME is not None:
    ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME
    fold_dirs = sorted(p.parent.name for p in ckpt_dir.glob("*/best.pt")) if ckpt_dir.exists() else []
    if fold_dirs:
        representative_model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES[TASK], num_speakers=1).to(device)
        load_checkpoint(ckpt_dir / fold_dirs[0] / "best.pt", representative_model, map_location=str(device))
        representative_model.eval()

representative = save_representative_utterance_artifacts(
    df_m6, model=representative_model, device=device, n_per_class=2, seed=config.DEFAULT_SEED)
representative[["Filename", "Speaker_ID", "Severity", "signal_figure"]]

In [ ]:
feature_architecture_table = build_feature_architecture_table()
model_dimensions_table = build_model_dimensions_table()
print_kv("Feature architecture table", "\n" + feature_architecture_table.to_string(index=False))
print_kv("Model dimensions table", "\n" + model_dimensions_table.to_string(index=False))

In [ ]:
if RUN_NAME is not None and preds is not None:
    for embedding_type in ("fused", "learned"):
        try:
            plot_embedding_map(RUN_NAME, preds, task=TASK, method="tsne", embedding_type=embedding_type,
                               color_by="speaker", show=True)
        except (FileNotFoundError, ValueError) as e:
            print_status(f"{embedding_type} (speaker): {e}", ok=False)

per_fold_path = config.METRICS_DIR / RUN_NAME / f"{RUN_NAME}.per_fold.csv" if RUN_NAME else None
if per_fold_path is not None and per_fold_path.exists():
    per_fold_df = pd.read_csv(per_fold_path)
    if "speaker_accuracy" in per_fold_df.columns:
        print_kv("Speaker-head accuracy (mean +/- std)",
                 f"{per_fold_df['speaker_accuracy'].mean():.3f} +/- {per_fold_df['speaker_accuracy'].std():.3f}")
        print_status("A speaker classifier near chance accuracy on the fused representation "
                    "suggests reduced speaker-specific information -- a diagnostic, not proof "
                    "of invariance; visual separation above is a representation diagnostic too.",
                    ok=True)

In [ ]:
complementarity_table = None
if RUN_NAME is not None:
    try:
        z_l = np.vstack(load_run_embeddings(RUN_NAME, embedding_type="learned")["embedding"].to_numpy())
        z_s = np.vstack(load_run_embeddings(RUN_NAME, embedding_type="segmental")["embedding"].to_numpy())
        z_p = np.vstack(load_run_embeddings(RUN_NAME, embedding_type="supra")["embedding"].to_numpy())
        n = min(len(z_l), len(z_s), len(z_p))
        _, complementarity_table = plot_complementarity_heatmap(z_l[:n], z_s[:n], z_p[:n], show=True)
    except (FileNotFoundError, ValueError) as e:
        print_status(str(e), ok=False)
complementarity_table

In [ ]:
written_tables = export_paper_tables(
    RUN_NAME or "severity_gated_fusion_three_branch", manifest=df_m6, ablation_df=ablation_df,
    gate_summary_df=gate_summary_df, shap_group_df=shap_group_df,
    permutation_group_df=permutation_group_df, task=TASK)

for name, path in written_tables.items():
    print_kv(name, path)

print_header("REAL ONE-SHOT TRAINING HAS NOT BEEN EXECUTED" if RUN_NAME is None else
            f"Final results reflect run: {RUN_NAME}")
written_tables.get("limitations")